In [ ]:
import os
import glob
import numpy as np
import torch
import pandas as pd
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import json
from raven.model import GPTConfig, GPT
from tqdm import tqdm
from raven.dataset import UnifiedSeqEHRDataset

%load_ext autoreload
%autoreload 2

In [ ]:
# ── Configurable paths ──
MODEL_BASE_DIR = "./output"  # Path to model output directory
MODEL_RUN_DIR = os.path.join(MODEL_BASE_DIR, "scaling_v2/out_multi_win_True_standard_grad_clip_1.0_lr_0.00022_min_lr_2.2e-05_top_perc_0_n_layer_8_n_head_8_n_embd_512_rotary_embedding_True_use_xpos_False_agg_labels_False_reverse_temporal_decay_0.75_block_size_512")

# you only need to change MODEL_RUN_DIR above to point to the right model
file1 = os.path.join(MODEL_RUN_DIR, "result_ontime_0_sum_sum_f1_all_condition_True_600000.json")
file2 = os.path.join(MODEL_RUN_DIR, "result_ontime_1_sum_sum_f1_all_condition_True_600000.json")

# 1. Load both JSON files
with open(file1, "r") as f:
    data1 = json.load(f)
with open(file2, "r") as f:
    data2 = json.load(f)

# 2. Merge them
merged_data = {}

# First, copy everything from data1 into merged_data
for condition, info in data1.items():
    merged_data[condition] = {
        "on_time": info.get("on_time", []),
        "pred_day": info.get("pred_day", []),
        "gt_day": info.get("gt_day", [])
    }

# Then, extend with entries from data2
for condition, info in data2.items():
    if condition not in merged_data:
        # If this condition wasn't in data1, just copy it
        merged_data[condition] = {
            "on_time": info.get("on_time", []),
            "pred_day": info.get("pred_day", []),
            "gt_day": info.get("gt_day", [])
        }
    else:
        # If it already exists, append the lists
        merged_data[condition]["on_time"].extend(info.get("on_time", []))
        merged_data[condition]["pred_day"].extend(info.get("pred_day", []))
        merged_data[condition]["gt_day"].extend(info.get("gt_day", []))

In [ ]:
# We'll create a list of rows for the DataFrame
rows = []

for disease, info in merged_data.items():
    on_time_values = info.get("on_time", [])
    
    # Convert them to Python booleans if they come in as "true"/"false" strings.
    # Depending on your JSON, they might already be Python True/False. 
    # If they're strings, you can do: 
    # on_time_values = [x is True or x == "true" for x in on_time_values]
    
    total_count = len(on_time_values)
    true_count = sum(on_time_values)  # sum(True/False) gives the count of Trues in Python
    on_time_rate = true_count / total_count if total_count > 0 else 0
    
    row = {
        "disease": disease,
        "total_count": total_count,
        "true_count": true_count,
        "on_time_rate": on_time_rate
    }
    rows.append(row)

# Convert the list of rows into a DataFrame
df = pd.DataFrame(rows)

# Sort by on_time_rate descending (optional)
df = df.sort_values(by="on_time_rate", ascending=False)

# Print or save the results
print(df)